# 📝 과제 01: 컴퓨터 기기 상품 조회와 구매 상담

## 만들 결과

가상 쇼핑몰 **모아디지털**의 구매 전 상담 도우미를 만듭니다. 가격·재고·기능은 상품 DB에서 확인하고, 사용 방법·호환 조건·쇼핑몰 안내는 문서에서 찾습니다. 질문에 필요한 소스를 골라 실제 상품 사실과 일반 안내를 구분해 답하세요.

- 상품번호 여러 개를 한 번에 조회하거나, 상품 종류·한 개당 최대 가격·재고·무선 조건으로 후보를 찾습니다.
- 일반 안내, 상품 조회, 조회 후 문서 검색, 추가 질문을 구분합니다.
- 상품 근거와 검색 기록을 보존하고, 답변이 어느 자료를 사용했는지 확인합니다.

교안01의 Adaptive 흐름을 상품 상담에 적용합니다. 질문에 따라 필요한 자료를 선택하는 것이 핵심이며, 조건 검색용 SQL 보조 함수는 제공합니다.

<img src="images/shop_assignment01_overview.png" width="1050" alt="상품 DB와 사용 안내 문서를 질문에 따라 선택하고 근거를 구분해 답합니다.">

## 준비: 데이터와 제공 코드

상품은 **30개 모델의 색상별 SKU 120건**입니다. SKU는 색상 등 판매 단위별로 구분한 상품입니다. 키보드·마우스·USB허브·모니터·웹캠·헤드셋이 각각 20건 있습니다. `products/products.csv`와 SQLite의 `products` 테이블에는 같은 가격·재고·연결 방식·지원 OS·기능 제한이 들어 있습니다.

문서는 **120개 중 현행 108개**를 검색합니다. 사용 안내 48개, 문제 해결 FAQ 36개, 쇼핑몰 정책 12개, 상품별 안내 12개와 폐지 문서 12개입니다. [과제 자료 안내](data/과제/)에서 실제 자료를 먼저 살펴보세요. 상품명·가격·재고·정책은 수업용 가상 설정입니다.

다음 순서로 진행하세요: **State 정의 → 계획 노드 → 상품 조회 노드 → 문서 검색 노드 → 답변·라우팅 함수 → 그래프 연결**. 환경 설정·하이브리드 검색·SQL 조회·상품 문서 변환 코드는 제공합니다. 작성 셀의 `...`에는 해당 단계의 코드를 넣으세요.

| 코드의 역할 | 무엇을 하나요? | 반환하는 값 |
|---|---|---|
| 노드 | 질문을 판단하거나 조회·검색·답변을 수행합니다. | State에 반영할 필드와 값이 담긴 dict |
| 라우팅 함수 | 앞 노드의 결과를 읽고 다음 노드를 고릅니다. | 다음 노드의 이름인 문자열 |
| 보조 함수 | DB 조회나 근거 문서 변환처럼 노드 안에서 필요한 일을 합니다. | 각 함수에 안내된 자료형 |
| 초기화 함수 | 새로운 질문에 사용할 State를 만듭니다. | 필요한 필드가 모두 들어 있는 새 dict |

노드에서는 입력 `state`를 직접 바꾸지 말고, 변경할 필드만 새 dict로 반환하세요. 그래프가 이 값을 State에 반영합니다. 라우팅 함수는 노드가 아니며 LLM을 호출하거나 State를 바꾸지 않습니다.

각 절의 **반환 예시**는 자료형과 데이터 구성을 보여 줍니다. 예시에 적은 입력·조회·검색 결과를 가정하며, 모델 문장과 검색 순위는 실제 실행마다 달라질 수 있습니다.

작성 후 바로 아래 자가채점 셀을 실행하세요. 자가채점은 고정된 모델 응답이나 로컬 DB로 확인하므로 외부 API를 호출하지 않습니다. 모든 작성이 끝난 뒤 마지막의 실제 문의 셀을 하나씩 실행하고, 검색 기록과 최종 답변을 읽습니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

In [ ]:
import os
from pathlib import Path
from typing import Literal, TypedDict
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown, display
from langgraph.graph import StateGraph, START, END

# 정답 폴더에서도 같은 단원의 data를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data" / "과제"
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"), use_responses_api=True)
# 배포 문서 벡터와 동일한 모델·차원으로 검색 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                  check_embedding_ctx_length=False)  # 앞 단원처럼 자동 길이 검사·분할을 끕니다.
chroma_dir = data_dir / "knowledge" / "chroma"

In [ ]:
def evidence_id(doc):
    """문서나 상품 조회 결과에 부여한 근거 ID를 반환합니다."""
    return doc.metadata["source_id"]


vector_store = Chroma(collection_name="day54_shop_documents", persist_directory=str(chroma_dir),
                      embedding_function=embedding_model, create_collection_if_not_exists=False)
# 저장된 본문·메타데이터를 읽습니다. 문서를 다시 임베딩하지 않습니다.
stored_documents = vector_store.get(include=["documents", "metadatas"])
documents = [
    Document(id=doc_id, page_content=text, metadata={**meta, "source_type": "knowledge"})
    for doc_id, text, meta in zip(stored_documents["ids"], stored_documents["documents"],
                                 stored_documents["metadatas"], strict=True)
]
print("검색 문서:", len(documents), "/ 검색 대상:", sum(doc.metadata["active"] for doc in documents))
display(pd.DataFrame([{"ID": evidence_id(doc), "제목": doc.metadata["title"],
                       "검색 사용": doc.metadata["active"]} for doc in documents[:6]]))

In [ ]:
kiwi = Kiwi()


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    # 문서와 질문에 같은 토큰화 기준을 적용해 BM25가 단어를 비교하게 합니다.
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


# 같은 현행 문서 집합의 BM25와 의미 검색을 가중 RRF로 묶습니다.
bm25 = BM25Retriever.from_documents(
    [doc for doc in documents if doc.metadata["active"]],
    preprocess_func=kiwi_tokenize, k=12,
)
dense = vector_store.as_retriever(search_kwargs={"k": 12, "filter": {"active": True}})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")


def search_documents(search_query, top_k=6):
    """하이브리드 순위에서 최대 top_k개를 반환합니다."""

    # 내부 검색은 각각 12개입니다. top_k는 결합한 뒤 전달할 수를 제한합니다.
    return hybrid.invoke(search_query)[:top_k]

In [ ]:
# [제공 코드] 자가채점은 고정 응답으로 확인하며 외부 API를 호출하지 않습니다.
from copy import deepcopy
from math import isclose
from types import SimpleNamespace
from typing import is_typeddict
from unittest.mock import Mock, patch, call

## 1. 질문별 State를 설계하세요

**작성:** 상태 구조 `RAGState`와 초기화 함수 `make_initial_state(question, top_k=6)`를 설계하세요. 두 대상은 그래프에 등록하는 노드가 아닙니다.

**목적:** 질문별로 계획·조회·검색·답변을 저장할 새 State를 만듭니다. 아래 필드명과 타입을 사용하세요.

| 구분 | 필드와 의미 |
|---|---|
| 입력 | `question`: 원질문 문자열, `top_k`: 문서 검색 한 번에서 받을 최대 문서 수 |
| 계획 | `route`: 처리 경로, `reason`: 경로 선택 이유, `search_query`: 문서 검색어 |
| 상품 조회 조건 | `product_ids`: 상품번호 문자열 목록, `category`: 상품 분류 문자열, `max_price`: 상품 한 개의 최대 가격(정수 또는 `None`) |
| 상품 조회 조건 | `in_stock`: 재고가 있는 상품만 찾을지 여부, `wireless`: 무선 상품만 찾을지 여부(각각 bool) |
| 추가 안내 | `clarification`: 사용자에게 물을 추가 질문, `product_notice`: 상품이 조회되지 않았을 때의 안내 |
| 근거·기록 | `documents`: 상품·문서 근거인 `Document` 목록, `search_log`: 조회·검색 기록 dict 목록 |
| 답변 | `draft`: 최종 답변 문자열, `evidence_ids`: 답변에서 인용한 근거 ID 문자열 목록 |

**구현할 처리**

1. `RAGState`를 `TypedDict`로 정의하세요. `route`에는 `knowledge`, `products`, `both`, `clarify` 중 하나가 들어갑니다.
2. 초기화 함수에서는 전달받은 `question`과 `top_k`를 저장하세요. 아직 판단하기 전의 `route`는 `clarify`로 시작합니다.
3. `max_price`는 `None`, `in_stock`과 `wireless`는 `False`로 시작하세요. `False`는 해당 조건으로 제한하지 않는다는 뜻이며, 품절 상품이나 유선 상품만 찾겠다는 뜻이 아닙니다.
4. 나머지 문자열은 빈 문자열, 목록은 새 빈 목록으로 만들고 전체 State를 반환하세요. 호출할 때마다 새 목록을 만들어 서로 다른 질문의 기록이 섞이지 않게 합니다.

**반환값: `dict`** — `make_initial_state`는 위 표의 모든 필드를 모두 담은 초기 상태를 반환합니다. DB 경로나 상품 전체 목록은 State에 추가하지 않습니다.

**반환 예시:** `make_initial_state("KB-102의 가격은?", top_k=4)`

```python
{
    "question": "KB-102의 가격은?", "top_k": 4,
    "route": "clarify", "reason": "", "search_query": "",
    "product_ids": [], "category": "", "max_price": None,
    "in_stock": False, "wireless": False,
    "clarification": "", "product_notice": "",
    "documents": [], "search_log": [], "draft": "", "evidence_ids": []
}
```

In [ ]:
# [상태 구조] 각 노드가 읽고 갱신할 필드를 정의합니다.
class RAGState(TypedDict):
    # 1. 입력, 계획, 조회 조건, 안내, 근거·기록, 답변 필드의 타입을 선언하세요.
    ...


# [초기화 함수] 새로운 질문의 작업 상태를 만듭니다.
def make_initial_state(question, top_k=6) -> RAGState:
    """입력값과 각 필드의 초기값을 담은 새 State를 반환합니다."""
    # 2. 입력 질문·검색 상한을 담고 초기 경로를 정하세요.
    ...
    # 3. 가격 조건과 두 bool 조건의 초기값을 정하세요.
    ...
    # 4. 빈 문자열·새 목록을 포함한 전체 State를 반환하세요.
    ...

In [ ]:
# 별도 질문이 같은 가변 목록을 공유하지 않는지 확인합니다.
first = make_initial_state("첫 문의", top_k=4)
second = make_initial_state("둘째 문의")
assert is_typeddict(RAGState), "RAGState를 TypedDict로 정의하세요."
assert first["question"] == "첫 문의" and first["top_k"] == 4
assert first["route"] == "clarify" and first["max_price"] is None
assert not first["in_stock"] and not first["wireless"]
first["documents"].append("확인용")
assert second["documents"] == [], "질문끼리 근거 목록이 공유되지 않아야 합니다."
print("초기 상태 확인 완료")

## 2. 질문에 필요한 경로와 상품 조건을 정하세요

이번에 만들 `route_query`는 **계획을 만드는 노드**입니다. 이름에 `route`가 있지만 다음 노드 이름을 바로 반환하는 라우팅 함수는 아닙니다. 이 노드가 State에 저장한 `route`를 나중에 `route_after_plan` 라우팅 함수가 읽습니다.

| 질문에 필요한 정보 | `route` 값 | 이어서 할 처리 |
|---|---|---|
| 일반 사용 안내·쇼핑몰 정책 | `knowledge` | 문서 검색 |
| 상품번호 또는 가격·재고 등의 조건에 맞는 상품 정보 | `products` | 상품 DB 조회 |
| 특정 상품의 사양과 사용 방법·적용 조건 | `both` | 상품 DB 조회 후 문서 검색 |
| 상품 또는 요청을 특정할 수 없음 | `clarify` | 사용자에게 추가 질문 |

제공된 `ProductPlan`은 LLM 응답의 형식이고, `plan_chain`은 질문을 그 형식으로 판단하는 체인입니다. 최대 가격은 **상품 한 개의 가격**입니다. 무선 조건에는 Bluetooth와 2.4GHz를 모두 포함합니다. 상품번호가 있으면 지정한 상품을 조회하며, 번호가 없을 때만 분류·가격·재고·무선 조건으로 목록을 찾습니다.

<img src="images/node_flow/shop_adaptive_route_query.png" width="1050" alt="과제 전체 adaptive 흐름에서 route_query 노드만 강조합니다.">

In [ ]:
import re

class ProductPlan(BaseModel):
    route: Literal["knowledge", "products", "both", "clarify"] = Field(description=(
        "일반 사용법·쇼핑몰 정책만 필요하면 knowledge, 상품 DB의 가격·재고·사양만 필요하면 products, "
        "특정 상품 사양과 문서의 사용법·조건을 함께 확인해야 하면 both, 대상이나 요청 확인이 필요하면 clarify입니다."))
    reason: str = Field(description="선택한 경로가 필요한 이유를 사용자 질문과 연결해 한 문장으로 설명합니다.")
    category: Literal["", "키보드", "마우스", "USB허브", "모니터", "웹캠", "헤드셋"] = Field(description="질문에 명시된 상품 종류이며, 없는 경우 빈 문자열입니다.")
    max_price: int | None = Field(description="질문에 명시된 상품 한 개의 최대 가격(원)입니다. 가격 조건이 없으면 null입니다. 총 구매 예산을 단가로 바꾸지 않습니다.")
    in_stock: bool = Field(description="사용자가 재고 있는 상품만 요청했을 때만 true입니다.")
    wireless: bool = Field(description="사용자가 무선 상품만 요청했을 때만 true입니다. 무선은 Bluetooth 또는 2.4GHz 연결입니다.")
    search_query: str = Field(description="knowledge 또는 both의 문서 검색 표현입니다. 질문에 명시된 대상·조건을 남기고 상품번호와 추측한 사양·답은 제외합니다. 다른 경로는 빈 문자열입니다.")
    clarification: str = Field(description="clarify에서 필요한 상품번호·종류·사용 환경 등을 묻는 질문입니다. 다른 경로는 빈 문자열입니다.")

plan_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자 질문에 필요한 정보 종류를 보고 경로와 검색 조건을 선택하세요. "
     "가격·재고·기본 사양은 products, 일반 사용법·FAQ·쇼핑몰 안내는 knowledge입니다. "
     "특정 상품의 지원 여부와 설정 방법·호환 조건을 함께 물으면 both입니다. "
     "상품번호로 상품을 지정하거나 종류·가격·재고·무선 조건으로 목록을 조회할 수 있습니다. "
     "질문의 최대 가격은 한 개의 상품 가격이며, 없는 조건을 추가하지 마세요. "
     "상품번호가 명시된 질문에는 그 번호의 사양과 안내를 확인하고 다른 모델로 바꾸지 마세요. "
     "이 상품·그 제품처럼 대상이 불명확하면 clarify로 상품번호를 물으세요. "
     "주문·결제·배송 추적 기록은 없으므로 특정 주문 상태나 도착일은 조회할 수 없습니다. "
     "검색 전에는 상품의 기능·가격·재고를 추측하지 말고, 문의와 관련된 검색어만 작성하세요."),
    ("human", "{question}"),
])
plan_chain = plan_prompt | llm.with_structured_output(ProductPlan)

**작성:** `route_query(state)` 노드를 설계하세요.

**목적:** 고객 질문을 처리할 경로와 상품 조회·문서 검색 조건을 결정합니다.

**구현할 처리**
1. 원질문을 `question`에 담아 `plan_chain`을 한 번 호출합니다. 결과는 `ProductPlan` 객체이므로 `plan.route`처럼 속성으로 사용합니다.
2. 정규표현식으로 상품번호를 추출하고, 대문자로 통일한 뒤 첫 등장 순서로 중복을 제거합니다.
3. `products`·`both`인데 번호와 조회 조건이 모두 없으면 `clarify`로 바꾸고 이유와 추가 질문을 작성합니다. 조건이 없다는 것은 `category=''`, `max_price=None`, `in_stock=False`, `wireless=False`인 경우입니다. 가격 `0`은 `None`과 구분합니다.
4. 최종 경로에 맞춰 아래 **9개 필드를 담은 새 dict**를 반환합니다. 입력 State는 수정하지 않습니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `route` | 최종 경로: `knowledge`, `products`, `both`, `clarify` 중 하나 |
| `reason` | 최종 경로를 선택한 이유 문자열 |
| `clarification` | `clarify`일 때 사용자에게 물을 질문. 그 외에는 계획의 빈 문자열 |
| `product_ids` | `products`·`both`이면 추출한 상품번호 목록, 그 외에는 `[]` |
| `category`, `max_price`, `in_stock`, `wireless` | 계획에서 받은 네 조회 조건을 각각 같은 이름의 키에 저장 |
| `search_query` | `knowledge`·`both`이면 계획 검색어의 앞뒤 공백을 제거한 문자열. 비면 원질문 사용. 그 외에는 `''` |

**반환 예시:** 질문이 “KB-102의 가격과 기기 등록·전환 방법을 알려 주세요.”이고, 계획이 `both`인 경우입니다.

```python
{
    "route": "both",
    "reason": "상품 가격과 사용 방법을 함께 확인해야 합니다.",
    "clarification": "",
    "product_ids": ["KB-102"],
    "category": "", "max_price": None,
    "in_stock": False, "wireless": False,
    "search_query": "기기 등록·전환 방법"
}
```

**상품번호 추출 도움말:** 제공된 `re` 모듈의 `re.findall`과 아래 패턴을 사용하세요. 원질문을 임시로 `.upper()`로 바꾸면 소문자 번호도 찾을 수 있고, `dict.fromkeys`로 추출 순서를 유지하며 중복을 제거할 수 있습니다.

```python
r"(?<![A-Z0-9-])(?:KB|MS|HB|MN|WC|HS)-[0-9]{3}(?![A-Z0-9-])"
```

가운데는 분류 코드·하이픈·숫자 3자리이며, 앞뒤 조건은 더 긴 영문·숫자·하이픈 문자열의 일부를 제외합니다. `kb-102와 KB-102, MS-203`은 `['KB-102', 'MS-203']`이 됩니다. `XKB-102`, `KB-1020`, `KB-102-A`는 제외합니다. 번호의 실제 존재 여부는 다음 조회 노드에서 확인합니다.

In [ ]:
# [노드] 질문에 필요한 경로와 조회·검색 계획을 만듭니다.
def route_query(state: RAGState):
    """원질문을 읽어 계획 필드의 갱신값을 dict로 반환합니다."""
    # 1. 원질문으로 plan_chain을 한 번 호출하고 계획 값을 읽으세요.
    ...
    # 2. 정규표현식으로 번호를 추출하고 대문자·첫 등장 순서로 정리하세요.
    ...
    # 3. 조회할 번호와 조건이 모두 없으면 clarify로 바꾸고 이유·질문을 작성하세요.
    ...
    # 4. 반환 표의 9개 계획 필드를 담은 dict를 반환하세요.
    ...

In [ ]:
# 고정 계획으로 번호 중복 제거와 조회 대상 누락 처리를 확인합니다.
plan = SimpleNamespace(route="both", reason="사양과 사용법 필요", category="", max_price=None,
                       in_stock=False, wireless=False, search_query="기기 전환", clarification="")
state = make_initial_state("kb-102와 KB-102, KB-103의 기기 전환을 비교해 주세요.")
before = deepcopy(state)
with patch.dict(globals(), {"plan_chain": Mock(invoke=Mock(return_value=plan))}):
    planned = route_query(state)
    missing = route_query(make_initial_state("이 상품은 어떻게 쓰나요?"))
    assert planned["product_ids"] == ["KB-102", "KB-103"]
    assert planned["route"] == "both" and planned["search_query"] == "기기 전환"
    assert missing["route"] == "clarify" and missing["clarification"]
    assert state == before, "입력 State를 수정하지 마세요."
print("계획과 상품번호 확인 완료")

## 3. 상품 조회 결과를 공통 근거로 바꾸세요

이번 노드는 `route_query`가 저장한 번호 또는 조건으로 상품 DB를 조회합니다. 아래 두 보조 함수는 완성 코드로 제공하므로, 역할과 인자를 확인한 뒤 노드 안에서 사용하세요.

| 제공 보조 함수 | 입력과 반환값 |
|---|---|
| `find_products(product_ids, category='', max_price=None, in_stock=False, wireless=False)` | 번호 또는 네 조건을 받아 상품 한 건당 dict 하나인 목록을 반환합니다. 결과가 없으면 빈 목록입니다. |
| `product_document(record)` | 상품 dict 한 건을 답변의 근거로 사용할 `Document` 한 개로 바꿉니다. 메타데이터의 `product_id`는 상품번호, `product`는 상품명입니다. |

번호가 있으면 지정한 번호들을 SQL 한 번으로 조회하며, 없는 번호는 반환 목록에 포함되지 않습니다. 번호가 없으면 조건 검색을 합니다. **조건 검색 결과는 가격 오름차순, 같은 가격에서는 상품번호 오름차순으로 최대 6개**입니다. 이 결과를 전체 상품이나 유일한 후보라고 표현하지 마세요.

<img src="images/node_flow/shop_adaptive_lookup_products.png" width="1050" alt="과제 전체 adaptive 흐름에서 lookup_products 노드만 강조합니다.">

In [ ]:
import sqlite3
from contextlib import closing

product_db_path = data_dir / "products" / "products.sqlite3"
# 상품 번호·가격·재고·사양이 어떤 형식인지 살펴봅니다.
display(pd.read_csv(data_dir / "products" / "products.csv").head(6))


# [보조 함수·제공] SQL 한 번으로 번호 조회 또는 조건 검색을 수행합니다.
def find_products(product_ids, category="", max_price=None, in_stock=False, wireless=False):
    """번호 조회 또는 조건 검색을 수행합니다. 조건 검색은 가격순 최대 6개입니다."""
    conditions = ["active = 1"]
    params = []
    # 번호가 있으면 해당 상품을 모두 조회합니다. 조건 검색의 상한과 필터를 적용하지 않습니다.
    if product_ids:
        placeholders = ", ".join(["?"] * len(product_ids))
        conditions.append(f"product_id IN ({placeholders})")
        params.extend(product_ids)
        ending = " ORDER BY product_id"
    else:
        if category:
            conditions.append("category = ?")
            params.append(category)
        if max_price is not None:
            conditions.append("price <= ?")
            params.append(max_price)
        if in_stock:
            conditions.append("stock > 0")
        if wireless:
            conditions.append("(connection LIKE '%Bluetooth%' OR connection LIKE '%2.4GHz%')")
        ending = " ORDER BY price, product_id LIMIT 6"
    # SQL은 정해진 조건만 조합하고, 사용자의 값은 매개변수로 전달합니다.
    sql = "SELECT * FROM products WHERE " + " AND ".join(conditions) + ending
    uri = product_db_path.resolve().as_uri() + "?mode=ro"
    with closing(sqlite3.connect(uri, uri=True)) as connection:
        connection.row_factory = sqlite3.Row
        rows = connection.execute(sql, params).fetchall()
    return [dict(row) for row in rows]


# [보조 함수·제공] 상품 한 건을 다른 근거와 같은 Document 형식으로 바꿉니다.
def product_document(record):
    """상품 사실을 답변과 문서 검색에서 사용할 근거로 바꿉니다."""
    doc_id = f"catalog:{record['product_id']}"
    text = (f"상품번호: {record['product_id']}\n상품명: {record['name']}\n분류: {record['category']}\n"
            f"가격: {record['price']}원\n재고: {record['stock']}개\n연결 방식: {record['connection']}\n"
            f"기본 지원 OS: {record['compatible_os']}\nBluetooth 등록 가능 대수: {record['multi_device']}\n"
            f"보증: {record['warranty_months']}개월\n기능·제약: {record['specification']}")
    return Document(id=doc_id, page_content=text, metadata={
        "source_id": doc_id, "title": record["name"], "source": "모아디지털 가상 상품 카탈로그",
        "source_type": "catalog", "product_id": record["product_id"], "product": record["name"],
    })

**작성:** `lookup_products(state)` 노드를 설계하세요.

**목적:** 상품번호 또는 조건으로 상품을 조회하고, 답변에 사용할 상품 근거와 미조회 안내를 만듭니다.

**구현할 처리**
1. State의 `product_ids`, `category`, `max_price`, `in_stock`, `wireless`를 `find_products`에 전달해 한 번 조회합니다.
2. 조회된 상품 dict를 각각 `product_document`로 변환합니다.
3. 요청 번호 중 없는 번호는 번호 확인 안내를, 조건 검색 결과가 없으면 조건 확인 안내를 작성합니다. 정상 조회이면 빈 문자열로 둡니다. 일부 번호만 없을 때 찾은 상품은 유지합니다.
4. 조회 기록 한 건에 `source='products'`, 실제 조회 조건인 `query`, 반환된 상품 근거의 `candidate_ids`를 담습니다. `query`는 번호 조회 시 번호 목록, 조건 검색 시 네 조건을 담은 dict입니다. 결과가 없어도 기록을 남깁니다.
5. 기존 근거·기록 뒤에 이번 결과를 추가한 새 목록으로 아래 dict를 반환합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `documents` | 기존 근거 뒤에 이번 상품 `Document`들을 추가한 목록 |
| `search_log` | 기존 기록 뒤에 이번 조회 기록 한 건을 추가한 목록 |
| `product_notice` | 미조회 안내 문자열. 정상 조회이면 `''` |

**반환 예시:** 기존 근거·기록이 비어 있고 `product_ids=["KB-102", "KB-999"]`인 경우입니다. `product_doc`는 조회된 KB-102를 `product_document`로 변환한 **Document 객체**이며, 근거 ID는 `catalog:KB-102`입니다. KB-999는 조회되지 않습니다.

```python
{
    "documents": [product_doc],
    "search_log": [{
        "source": "products",
        "query": ["KB-102", "KB-999"],
        "candidate_ids": ["catalog:KB-102"]
    }],
    "product_notice": "조회되지 않은 상품번호: KB-999. 번호를 확인해 주세요."
}
```

`documents`에는 ID 문자열이 아니라 본문·메타데이터를 가진 객체를 넣습니다. `candidate_ids`에는 그 객체들의 ID 문자열만 넣습니다.

In [ ]:
# [노드] 상품을 조회하고 공통 근거와 조회 기록을 만듭니다.
def lookup_products(state: RAGState):
    """상품 근거·조회 기록·미조회 안내의 갱신값을 반환합니다."""
    # 1. 번호 목록과 네 조건을 전달해 상품을 한 번 조회하세요.
    ...
    # 2. 조회된 각 상품을 Document로 바꾸세요.
    ...
    # 3. 일부 번호 미조회, 조건 검색 결과 없음, 정상 조회를 구분해 안내를 정하세요.
    ...
    # 4. 실제 조회 입력과 반환된 근거 ID로 기록 한 건을 만드세요.
    ...
    # 5. documents·search_log·product_notice를 담은 dict를 반환하세요.
    ...

In [ ]:
# 실제 로컬 DB로 조건 조회와 일부 번호 미조회를 확인합니다.
filtered = find_products([], category="마우스", max_price=50000, in_stock=True, wireless=True)
assert 0 < len(filtered) <= 6
assert all(r["category"] == "마우스" and r["price"] <= 50000 and r["stock"] > 0 for r in filtered)
state = make_initial_state("두 상품 확인")
state["product_ids"] = ["KB-102", "KB-999"]
before = deepcopy(state)
looked = lookup_products(state)
assert [evidence_id(d) for d in looked["documents"]] == ["catalog:KB-102"]
assert "KB-999" in looked["product_notice"] and len(looked["search_log"]) == 1
assert state == before
# 조건 전달과 기존 근거·로그 보존도 학생 노드에서 확인합니다.
state = make_initial_state("5만원 이하 무선 마우스 재고")
state.update(category="마우스", max_price=50000, in_stock=True, wireless=True)
old_doc = product_document(find_products(["KB-102"])[0])
old_log = {"source": "products", "query": ["KB-102"], "candidate_ids": ["catalog:KB-102"]}
state["documents"], state["search_log"] = [old_doc], [old_log]
before = deepcopy(state)
with patch.dict(globals(), {"find_products": Mock(wraps=find_products)}) as patched:
    filtered_result = lookup_products(state)
    assert patched["find_products"].call_count == 1
    assert [evidence_id(d) for d in filtered_result["documents"]] == ["catalog:KB-102"] + [f"catalog:{r['product_id']}" for r in filtered]
    assert filtered_result["search_log"][0] == old_log and not filtered_result["product_notice"]
    assert filtered_result["search_log"][1]["query"] == {"category": "마우스", "max_price": 50000, "in_stock": True, "wireless": True}
    assert state == before
print("조건 조회와 일부 미조회 처리 확인 완료")

## 4. 상품 근거를 유지하며 문서를 검색하세요

<img src="images/node_flow/shop_adaptive_search_knowledge.png" width="1050" alt="과제 전체 adaptive 흐름에서 search_knowledge 노드만 강조합니다.">

**작성:** `search_knowledge(state)` 노드를 설계하세요.

**목적:** 기존 상품 근거를 유지하면서 질문에 필요한 사용 안내 문서를 검색합니다.

**구현할 처리**
1. 기존 `documents`와 `search_log`를 새 목록으로 복사합니다. 이 시점의 근거는 상품 문서이거나 빈 목록입니다.
2. 상품 근거의 등장 순서대로 `product_id`·`product` 메타데이터와 `search_query`를 포함한 검색어를 만듭니다. 상품 근거가 없으면 `search_query`만 사용합니다.
3. 같은 검색어는 하나로 묶고 관련 상품번호 목록을 연결합니다. 일반 문서 검색의 상품번호 목록은 비웁니다.
4. 검색어가 만들어진 순서대로 `search_documents`를 한 번씩 호출하고 `top_k`를 전달합니다. 문서를 추가하고, `source='knowledge'`, `query`, 관련 `product_ids`, 검색 순서대로 된 `candidate_ids`를 기록합니다. ID는 `evidence_id(doc)`로 얻습니다.
5. 근거 문서만 ID로 중복 제거하고 검색 기록은 모두 보존합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `documents` | 기존 상품 근거와 검색 문서를 합친 뒤 ID 중복을 제거한 `Document` 목록 |
| `search_log` | 기존 조회 기록 뒤에 질의별 검색 기록을 추가한 목록 |

**반환 예시:** KB-102 상품 근거·조회 기록을 이미 가지고 있고, 검색어가 `기기 등록·전환 방법`인 경우입니다. 검색에서 `guide:KB-102-pairing` 한 개를 받았다고 가정합니다. `product_doc`는 기존 상품 Document, `guide_doc`는 검색된 안내 Document입니다.

```python
{
    "documents": [product_doc, guide_doc],
    "search_log": [
        {"source": "products", "query": ["KB-102"],
         "candidate_ids": ["catalog:KB-102"]},
        {"source": "knowledge",
         "query": "KB-102 모아 스위치75 블랙 / 기기 등록·전환 방법",
         "product_ids": ["KB-102"],
         "candidate_ids": ["guide:KB-102-pairing"]}
    ]
}
```

In [ ]:
# [노드] 상품 근거를 보존하면서 사용 안내 문서를 검색합니다.
def search_knowledge(state: RAGState):
    """문서 근거와 검색 기록의 갱신값을 반환합니다."""
    # 1. 기존 근거와 기록을 새 목록으로 복사하세요.
    ...
    # 2. 상품번호·상품명·계획 검색어를 결합하거나 계획 검색어만 사용하세요.
    ...
    # 3. 같은 검색어와 그 검색에 관련된 상품번호를 묶으세요.
    ...
    # 4. 검색어별로 한 번 검색하고 문서와 호출 기록을 추가하세요.
    ...
    # 5. ID 중복을 제거한 documents와 전체 search_log를 담은 dict를 반환하세요.
    ...

In [ ]:
# 같은 문서가 여러 검색에서 나와도 근거는 한 번만, 호출 기록은 모두 남깁니다.
state = make_initial_state("상품별 전환 방법")
state["search_query"] = "기기 전환 방법"
state["documents"] = [product_document(r) for r in find_products(["KB-102", "KB-103"])]
state["search_log"] = [{"source": "products", "query": ["KB-102", "KB-103"], "candidate_ids": ["catalog:KB-102", "catalog:KB-103"]}]
mock_doc = Document(page_content="전환 안내", metadata={"source_id": "check:guide"})
search = Mock(return_value=[mock_doc])
before = deepcopy(state)
with patch.dict(globals(), {"search_documents": search}):
    retrieved = search_knowledge(state)
    assert search.call_count == 2
    assert {evidence_id(d) for d in retrieved["documents"]} == {"catalog:KB-102", "catalog:KB-103", "check:guide"}
    assert len(retrieved["search_log"]) == 3 and state == before
    assert retrieved["search_log"][0] == before["search_log"][0]
    for index, doc in enumerate(before["documents"]):
        search_call = search.call_args_list[index]
        assert (search_call.kwargs["top_k"] if "top_k" in search_call.kwargs else search_call.args[1]) == state["top_k"]
        query = search_call.args[0] if search_call.args else search_call.kwargs["search_query"]
        assert all(part in query for part in [doc.metadata["product_id"], doc.metadata["product"], state["search_query"]])
        entry = retrieved["search_log"][index+1]
        assert entry == {"source": "knowledge", "query": query, "product_ids": [doc.metadata["product_id"]], "candidate_ids": ["check:guide"]}
print("상품 근거 보존과 문서 검색 확인 완료")

## 5. 답변과 조건부 경로를 완성하세요

답변의 구조화 출력·근거 표시·프롬프트는 제공합니다. `generate`는 교안의 답변 생성 패턴을 사용합니다. 답변에 상품 사실과 안내 문서가 구분되는지 읽으세요.

<img src="images/node_flow/shop_adaptive_generate.png" width="1050" alt="과제 전체 adaptive 흐름에서 generate 노드만 강조합니다.">

In [ ]:
class GroundedAnswer(BaseModel):
    answer: str = Field(description="원질문에 대한 답변입니다. 확인된 사실마다 [근거 ID]를 붙이고 확인되지 않은 항목을 구분합니다.")
    evidence_ids: list[str] = Field(description="답변에서 실제로 인용한 제공 근거 ID 목록입니다. 없는 ID를 만들지 않습니다.")


# [보조 함수·제공] 근거 ID·출처·본문을 답변 모델이 읽을 문자열로 만듭니다.
def format_documents(docs):
    return "\n\n".join(f"[{evidence_id(doc)}] {doc.metadata['title']}\n출처: {doc.metadata['source']}\n{doc.page_content}" for doc in docs)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 근거로만 원질문에 답하세요. 사실에는 [근거 ID]를 붙이고 실제 사용한 ID를 evidence_ids에 넣으세요. "
     "상품번호별 가격·재고·기능을 섞지 마세요. 상품 DB의 사실과 일반 사용 안내의 조건을 구분하세요. "
     "다른 상품의 설정키나 지원 기능을 문의한 상품에 적용하지 마세요. 일반 안내는 그 기능을 지원하는 제품에만 적용됩니다. "
     "Bluetooth 기기 등록 대수, 한 대씩 전환, 동시 입력은 서로 다릅니다. USB-C 단자만 보고 영상·충전·데이터 기능을 확정하지 마세요. "
     "원질문에 요청된 항목을 빠뜨리지 말고 근거가 없는 항목은 확인 불가로 밝혀 주세요. "
     "가격·재고는 catalog 근거에 있을 때만 답하고 실시간 재고라고 표현하지 마세요. "
     "조건 검색은 가격순 최대6개이므로 전체 상품이나 유일한 선택지라고 단정하지 마세요. "
     "조회되지 않은 상품 안내는 마지막 별도 문단에 적고 인용을 붙이지 마세요. "
     "조회 안내 문장은 사실 인용 규칙의 예외입니다. 다른 상품의 catalog ID로 미조회 사실을 뒷받침하지 마세요. "
     "지원 여부가 조건부이면 첫 문장부터 조건을 설명하고, 구매·배송·수리·호환을 확정 약속하지 마세요. "
     "간결하게 답하며 자료에 없는 가격·도착일·입고일·버튼 조합을 만들지 마세요."),
    ("human", "질문: {question}\n인용 가능한 ID: {allowed_ids}\n근거:\n{context}"),
])
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)

**부분 작성:** `generate(state)` 답변 생성 노드의 빈칸을 채우세요.

**목적:** 원질문에 대해 제공된 근거로 답변하고, 실제 인용한 근거 ID를 반환합니다.

**구현할 처리**
1. 근거가 없으면 LLM을 호출하지 않고 안내 문장과 빈 인용 목록을 반환합니다. 이 분기는 제공 코드입니다.
2. 각 문서의 `evidence_id`를 모아 `allowed_ids`를 완성합니다.
3. 제공된 호출 코드로 `answer_chain`에 원질문·인용 가능 ID·근거 본문·미조회 안내를 전달합니다.
4. 모델 응답을 아래 State 필드명에 맞춰 반환합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `draft` | 모델 응답의 `answer` 문자열. 근거가 없으면 안내 문장 |
| `evidence_ids` | 모델 응답의 인용 ID 문자열 목록. 근거가 없으면 `[]` |

**반환 예시:** KB-102의 가격 질문에 상품 근거로 답한 경우입니다. 모델의 `answer`가 State의 `draft`로 들어갑니다.

```python
{
    "draft": "KB-102의 가격은 59,000원입니다. [catalog:KB-102]",
    "evidence_ids": ["catalog:KB-102"]
}
```

In [ ]:
# [노드] 상품 사실과 안내 근거로 답변
def generate(state):
    # 1. 근거가 없으면 답변 모델을 호출하지 않고 안내합니다.
    if not state["documents"]:
        return {"draft": "제공 자료에서 답변 근거를 찾지 못했습니다.", "evidence_ids": []}
    # 2. 실제 제공하는 문서에서만 인용 가능한 ID를 모읍니다.
    allowed_ids = ...
    # 3. 검색어가 아닌 원질문과 준비된 근거·조회 안내로 답변을 생성합니다.
    answer = answer_chain.invoke({"question": state["question"], "allowed_ids": allowed_ids,
        "context": format_documents(state["documents"]) + "\n조회 안내: " + state.get("product_notice", "")})
    # 4. 모델 응답의 필드명을 State의 답변 필드명에 맞춰 반환합니다.
    return {"draft": ..., "evidence_ids": ...}

<img src="images/node_flow/shop_adaptive_respond.png" width="1050" alt="과제 전체 adaptive 흐름에서 respond 노드만 강조합니다.">

**작성:** `respond(state)` 노드와 라우팅 함수 두 개를 설계하세요.

**`respond(state)`: 추가 안내 노드**

**목적:** 미조회 안내나 추가 질문을 최종 응답으로 전달합니다.

**구현할 처리:** `product_notice` → `clarification` → 기본 안내 순으로 비어 있지 않은 문장을 선택하세요. 기본 안내는 상품번호 또는 원하는 조건을 묻는 문장입니다.

**반환값: `dict`**

| 키 | 값 |
|---|---|
| `draft` | 선택한 안내 문장 |
| `evidence_ids` | 빈 목록 `[]` |

**반환 예시:** `product_notice`가 아래 미조회 안내인 경우입니다.

```python
{
    "draft": "조회되지 않은 상품번호: KB-999. 번호를 확인해 주세요.",
    "evidence_ids": []
}
```

**`route_after_plan(state)`: 계획 뒤의 라우팅 함수**

**목적:** 계획의 `route`에 따라 다음 노드를 정합니다.

**반환값: `str`** — 아래 조건에 해당하는 노드 이름 하나입니다.

| 조건 | 반환 문자열 |
|---|---|
| `route == 'knowledge'` | `'search_knowledge'` |
| `route`가 `products` 또는 `both` | `'lookup_products'` |
| `route == 'clarify'` | `'respond'` |

**`route_after_products(state)`: 상품 조회 뒤의 라우팅 함수**

**목적:** 상품 조회 결과에 따라 다음 노드를 정합니다. 근거 유무를 먼저 확인하세요.

**반환값: `str`** — 아래 조건에 해당하는 노드 이름 하나입니다.

| 조건 | 반환 문자열 |
|---|---|
| `documents`가 비어 있음 | `'respond'` |
| 근거가 있고 `route == 'both'` | `'search_knowledge'` |
| 근거가 있고 `route == 'products'` | `'generate'` |

일부 상품번호만 조회돼도 근거가 있으면 두 번째 또는 세 번째 조건을 적용합니다. 세 함수 모두 State를 직접 수정하거나 LLM·DB·검색을 호출하지 않습니다.

In [ ]:
# [노드] 미조회 안내 또는 추가 질문을 최종 응답으로 돌려줍니다.
def respond(state: RAGState):
    # 1. 미조회 안내와 추가 질문을 읽으세요.
    ...
    # 2. 미조회 안내 → 추가 질문 → 기본 안내 순으로 사용할 문장을 고르세요.
    ...
    # 3. draft와 빈 evidence_ids를 담은 dict를 반환하세요.
    ...


# [라우팅 함수] route_query 노드 뒤에 실행할 노드를 고릅니다.
def route_after_plan(state):
    # 1. 계획에 저장된 route를 읽으세요.
    ...
    # 2. 경로 표에 맞는 다음 노드 이름을 문자열로 반환하세요.
    ...


# [라우팅 함수] lookup_products 노드 뒤에 실행할 노드를 고릅니다.
def route_after_products(state):
    # 1. 조회된 상품 근거가 있는지 먼저 확인하세요.
    ...
    # 2. 상품 근거가 하나도 없으면 추가 안내 노드 이름을 반환하세요.
    ...
    # 3. 근거가 있으면 both와 products를 구분해 다음 노드 이름을 반환하세요.
    ...

In [ ]:
# 경로 선택과 조회 결과의 경계 조건을 확인합니다.
state = make_initial_state("확인")
for route, expected in [("knowledge", "search_knowledge"), ("products", "lookup_products"), ("both", "lookup_products"), ("clarify", "respond")]:
    state["route"] = route
    assert route_after_plan(state) == expected
state["route"] = "both"
assert route_after_products(state) == "respond"
state["documents"] = [Document(page_content="상품 근거")]
assert route_after_products(state) == "search_knowledge"
state["route"] = "products"
assert route_after_products(state) == "generate"
print("조건부 경로 확인 완료")

## 6. 그래프를 연결하고 실제 문의를 확인하세요

**작성:** 지금까지 만든 노드와 라우팅 함수를 사용해 실행 그래프를 구성하세요. 각 함수를 정의한 셀과 자가채점 셀을 먼저 실행한 뒤 진행합니다.

**목적:** 질문에 따라 조회·검색·답변 경로를 선택하는 그래프를 만듭니다.

**구현할 처리**

1. `RAGState`를 사용하는 `StateGraph`를 만들어 `builder`에 저장하세요.
2. `route_query`, `lookup_products`, `search_knowledge`, `generate`, `respond` 다섯 함수를 같은 이름의 노드로 등록하세요. 라우팅 함수는 노드로 등록하지 않습니다.
3. 시작점 `START`를 `route_query`에 연결하세요. `route_query` 뒤에는 `route_after_plan`, `lookup_products` 뒤에는 `route_after_products`를 조건부 엣지로 연결하고, 각 라우팅 함수가 반환할 문자열과 대상 노드의 대응을 지정합니다.
4. `search_knowledge`에서 `generate`로, `generate`와 `respond`에서는 각각 `END`로 일반 엣지를 연결하세요.
5. 그래프를 컴파일해 `shopping_app`에 저장하세요. 제공된 표시 코드를 실행해 실제 연결과 각 노드 그림이 일치하는지 확인합니다. 컴파일만으로 고객 질문이 처리되는 것은 아닙니다.

**만들 결과:** 컴파일된 그래프 `shopping_app`입니다. 이 셀은 함수가 아니므로 반환문을 작성하지 않습니다.

아래 실제 문의는 각각 별도 셀입니다. `make_initial_state`로 새 상태를 만들고 `shopping_app.invoke`로 실행합니다. 매번 선택 경로와 이유 → 조회·검색 기록 → 최종 답변 순으로 확인하세요. 이 단계에서는 실제 API를 호출합니다.

In [ ]:
# 1. RAGState를 사용하는 그래프를 만드세요.
builder = ...
# 2. 다섯 노드를 이름과 함수로 등록하세요.
...
# 3. 시작점과 두 라우팅 함수의 조건부 엣지를 연결하세요.
...
# 4. 문서 검색 → 답변, 답변·추가 안내 → 종료를 연결하세요.
...
# 5. 컴파일한 그래프를 저장하고 연결을 표시하세요.
shopping_app = ...
display(Image(shopping_app.get_graph().draw_mermaid_png()))

**실행 확인 · 일반 안내**

In [ ]:
question = '배송비와 무료 배송 기준을 알려 주세요.'
result_policy = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_policy["route"], "/ 이유:", result_policy["reason"])
display(pd.DataFrame(result_policy["search_log"]))
display(Markdown(result_policy["draft"]))

**실행 확인 · 조건 검색**

In [ ]:
question = '5만 원 이하 무선 마우스 중 재고 있는 상품을 보여 주세요.'
result_filter = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_filter["route"], "/ 이유:", result_filter["reason"])
display(pd.DataFrame(result_filter["search_log"]))
display(Markdown(result_filter["draft"]))

**실행 확인 · 상품과 안내 결합**

In [ ]:
question = 'KB-102의 가격과 재고를 알려 주세요. 노트북과 태블릿에 번갈아 쓰려는데 지원 여부와 기기 등록·전환 방법도 설명해 주세요.'
result_combined = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_combined["route"], "/ 이유:", result_combined["reason"])
display(pd.DataFrame(result_combined["search_log"]))
display(Markdown(result_combined["draft"]))

**실행 확인 · 일부 번호 없음**

In [ ]:
question = 'KB-102와 KB-999의 가격과 재고를 각각 알려 주세요.'
result_partial = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_partial["route"], "/ 이유:", result_partial["reason"])
display(pd.DataFrame(result_partial["search_log"]))
display(Markdown(result_partial["draft"]))

**실행 확인 · 대상 확인**

In [ ]:
question = '이 상품은 아이패드에서도 되나요?'
result_clarify = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_clarify["route"], "/ 이유:", result_clarify["reason"])
display(pd.DataFrame(result_clarify["search_log"]))
display(Markdown(result_clarify["draft"]))

**실행 확인 · 번호 전체 없음**

In [ ]:
question = 'KB-999의 재고를 알려 주세요.'
result_missing = shopping_app.invoke(make_initial_state(question))
print("선택 경로:", result_missing["route"], "/ 이유:", result_missing["reason"])
display(pd.DataFrame(result_missing["search_log"]))
display(Markdown(result_missing["draft"]))

## 결과를 읽고 점검하세요

- 조건 검색이 무선·재고·최대 가격을 모두 만족하는지 상품 CSV와 대조합니다. 최대6개 결과를 전체 상품이라고 표현하지 않아야 합니다.
- KB-102의 가격·재고와 등록·전환 방법을 각각 상품 DB·설명서 근거로 답했는지 확인합니다. 여러 기기 등록과 동시 입력을 구분하세요.
- KB-999가 없을 때 다른 상품의 가격이나 안내를 붙이지 않는지 확인합니다.
- 모델 문장이나 검색 순위는 고정하지 않습니다. 인용한 ID의 원문이 실제 주장을 뒷받침하는지 읽으세요.